# Cortical Annotation Tool

## How to Use this Tool

This section should document how to use the annotation tool, generally.

## What to Annotate

This section should contain instructions for the specific contours or boundaries that are being drawn using the tool as part of the current annotation project.

The annotation tool is designed to be a general crowd-source tool for annotating the cortical surface. It is generally recommended to include some amount of instructions for anyone annotating cortices both to explain to the raters how annotations should be drawn and to document how the annotations were drawn for future researchers.

## The Annotation Tool

In [1]:
%%html
<!-- This HTML block inserts code into the notebook that hides the input
     blocks (such as this one) and the prompt panel (the space to the left 
     where "In [ ]" and "Out [ ]" appear).
-->
<script>
    $('div.prompt').hide();
    $('div.input').hide();
</script>

In [2]:
# This cell starts the Annotation Tool.
#
# All cells in this notebook are run auotmatically when the notebook starts in
# the docker container (if you are running this outside of the docker container
# you are on your own), and all input cells are automatically hidden from the
# user.
#
# First, we configure a few things (these are set up to work inside of the
# docker container).
config_path = '/config/config.yaml'
cache_path  = '/cache/tool'

import sys
sys.path.insert(0, '/src')

# Next, import the tool from the annotation library.
from annotate import AnnotationTool
# Then, we just start the tool.
# The tool displays itself during its init function (so that it can display a
# loading screen), so we capture this output.
tool = AnnotationTool(config_path=config_path, cache_path=cache_path)

Matplotlib created a temporary config/cache directory at /tmp/matplotlib-sjqtrdon because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


AnnotationTool(children=(ControlPanel(children=(HTML(value='\n            <style>\n            .jupyter-widget…

In [3]:
import sys, os
print(sys.path)
print(os.path.isdir('/src'))
print(os.listdir('/src') if os.path.isdir('/src') else 'NOT FOUND')

['/src', '/bcbl/home2/mmartinez/toolboxes/cortex-annotate/notebooks', '/src', '/opt/conda/lib/python310.zip', '/opt/conda/lib/python3.10', '/opt/conda/lib/python3.10/lib-dynload', '', '/opt/conda/lib/python3.10/site-packages', '/opt/conda/lib/python3.10/site-packages/setuptools/_vendor']
True
['annotate']


In [4]:
import neuropythy as ny
import numpy as np
import os

subjects_dir = '/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/freesurfer-with_t2'
prf_dir = '/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/prfanalyze-vista/analysis-02_one_gaussian'
prf_task = 'retCB'
prf_run = 'run-0102avg'
prf_session = 'ses-01'

sid = 'sub-01'
h = 'lh'
hemi_code = 'L'

fs_path = os.path.join(subjects_dir, sid)
sub = ny.freesurfer_subject(fs_path)
cortex = sub.hemis[h]
print("FreeSurfer surface vertex count:", cortex.vertex_count)

sub_prf_dir = os.path.join(prf_dir, sid, prf_session)
prefix = f'{sid}_{prf_session}_task-{prf_task}_{prf_run}_hemi-{hemi_code}_'
x0 = np.asarray(ny.load(os.path.join(sub_prf_dir, prefix + 'centerx0.nii.gz')))
print("Raw loaded x0 shape:", x0.shape)
print("Flattened x0 length:", x0.flatten().shape)

FreeSurfer surface vertex count: 128935
Raw loaded x0 shape: (25260,)
Flattened x0 length: (25260,)


In [5]:
import json, glob, os

maskinfo_dir = "/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/prfprepare/analysis-02_votcROI/sub-01/ses-01/func/"
files = glob.glob(os.path.join(maskinfo_dir, "sub-01_ses-01_hemi-L_desc-*_maskinfo.json"))
print("Number of ROI files found:", len(files))

all_indices = set()
for f in files:
    d = json.load(open(f))
    all_indices.update(d['roiIndFsnative'])

print("Union of all ROI vertex indices:", len(all_indices))
print("Expected (thisHemiSize):", 25260)

Number of ROI files found: 45
Union of all ROI vertex indices: 25260
Expected (thisHemiSize): 25260


In [6]:
import scipy.io as sio
mat = sio.loadmat(os.path.join(sub_prf_dir, prefix + 'results.mat'), simplify_cells=True)
results = mat['results']
print(type(results))
if isinstance(results, dict):
    print(list(results.keys()))
else:
    print(results)

<class 'dict'>
['model', 'params']


In [7]:
import json
d = json.load(open(os.path.join(sub_prf_dir, prefix + 'estimates.json')))
print(list(d[0].keys()))
print(len(d))

['Centerx0', 'Centery0', 'Theta', 'sigmaMajor', 'sigmaMinor', 'testdata', 'R2', 'modelpred']
25260


In [8]:
import nibabel as nib
bold = nib.load(os.path.join(maskinfo_dir, 'sub-01_ses-01_task-retCB_run-0102avg_hemi-L_bold.nii.gz'))
print(bold.shape)

(25260, 1, 1, 150)


In [9]:
import json, glob, os

maskinfo_dir = "/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/prfprepare/analysis-02_votcROI/sub-01/ses-01/func/"
files = sorted(glob.glob(os.path.join(maskinfo_dir, "sub-01_ses-01_hemi-L_desc-*_maskinfo.json")))

total_len = 0
seen = set()
overlap_found = False
for f in files:
    d = json.load(open(f))
    idx = d['roiIndFsnative']
    total_len += len(idx)
    for i in idx:
        if i in seen:
            overlap_found = True
        seen.add(i)

print("Number of files:", len(files))
print("Sum of individual ROI sizes:", total_len)
print("Union (unique) size:", len(seen))
print("Any overlapping vertices between ROIs?:", overlap_found)

Number of files: 45
Sum of individual ROI sizes: 40156
Union (unique) size: 25260
Any overlapping vertices between ROIs?: True


In [10]:
import json
d = json.load(open("/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/prfprepare/analysis-02_votcROI/sub-01/ses-01/func/sub-01_ses-01_hemi-L_desc-V1v-wang_maskinfo.json"))
print(list(d.keys()))
print("roiIndFsnative length:", len(d['roiIndFsnative']))
print("roiIndBold length:", len(d.get('roiIndBold', [])))
print("roiIndFsnative min/max:", min(d['roiIndFsnative']), max(d['roiIndFsnative']))
print("roiIndBold min/max:", min(d.get('roiIndBold', [0])), max(d.get('roiIndBold', [0])))

['atlas', 'roi', 'hemisphere', 'thisHemiSize', 'boldFiles', 'roiIndFsnative', 'roiIndBold']
roiIndFsnative length: 1246
roiIndBold length: 1246
roiIndFsnative min/max: 2511 128595
roiIndBold min/max: 2437 25191


In [11]:
import numpy as np, neuropythy as ny
d = '/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/prfanalyze-vista/analysis-02_one_gaussian/sub-01/ses-01/'
p = 'sub-01_ses-01_task-retCB_run-0102avg_hemi-L_'
x  = np.asarray(ny.load(d + p + 'centerx0.nii.gz')).flatten()
y  = np.asarray(ny.load(d + p + 'centery0.nii.gz')).flatten()
r2 = np.asarray(ny.load(d + p + 'r2.nii.gz')).flatten()
ecc = np.hypot(x, y)
good = r2 > 0.1
print("well-fit vertices:", good.sum())
print("eccentricity percentiles (50/90/99):", np.percentile(ecc[good], [50, 90, 99]))
print("max:", ecc[good].max())

well-fit vertices: 9969
eccentricity percentiles (50/90/99): [ 3.5557003  15.23996029 19.71850693]
max: 20.624826


In [12]:
import json, numpy as np, neuropythy as ny
d  = '/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/prfanalyze-vista/analysis-02_one_gaussian/sub-01/ses-01/'
md = '/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/prfprepare/analysis-02_votcROI/sub-01/ses-01/func/'
p  = 'sub-01_ses-01_task-retCB_run-0102avg_hemi-L_'
x  = np.asarray(ny.load(d + p + 'centerx0.nii.gz')).flatten()
y  = np.asarray(ny.load(d + p + 'centery0.nii.gz')).flatten()
r2 = np.asarray(ny.load(d + p + 'r2.nii.gz')).flatten()
ecc = np.hypot(x, y)

pos = []
for roi in ['V1v','V1d','V2v','V2d','V3v','V3d']:
    pos += json.load(open(md + f'sub-01_ses-01_hemi-L_desc-{roi}-wang_maskinfo.json'))['roiIndBold']
pos = np.array(sorted(set(pos)))
e, ok = ecc[pos], r2[pos] > 0.1
print("V1-V3 well-fit vertices:", ok.sum())
print("percentiles (50/75/90/95/99):", np.percentile(e[ok], [50, 75, 90, 95, 99]))
counts, edges = np.histogram(e[ok], bins=np.arange(0, 22, 1))
for c, lo in zip(counts, edges[:-1]):
    print(f"{lo:4.0f}-{lo+1:<4.0f} {c}")

V1-V3 well-fit vertices: 4609
percentiles (50/75/90/95/99): [ 4.54831648  8.59428692 16.59372139 18.62315025 20.09040733]
   0-1    193
   1-2    947
   2-3    502
   3-4    432
   4-5    441
   5-6    316
   6-7    248
   7-8    291
   8-9    183
   9-10   118
  10-11   62
  11-12   73
  12-13   63
  13-14   86
  14-15   54
  15-16   79
  16-17   107
  17-18   127
  18-19   112
  19-20   127
  20-21   48


In [13]:
import json, numpy as np, neuropythy as ny
sid = 'sub-11'
d  = f'/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/prfanalyze-vista/analysis-02_one_gaussian/{sid}/ses-01/'
md = f'/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/prfprepare/analysis-02_votcROI/{sid}/ses-01/func/'
for hemi in ['L', 'R']:
    p  = f'{sid}_ses-01_task-retCB_run-0102avg_hemi-{hemi}_'
    x  = np.asarray(ny.load(d + p + 'centerx0.nii.gz')).flatten()
    y  = np.asarray(ny.load(d + p + 'centery0.nii.gz')).flatten()
    r2 = np.asarray(ny.load(d + p + 'r2.nii.gz')).flatten()
    ok = (r2 > 0.1) & (np.hypot(x, y) < 9)
    for roi in ['V1v', 'V1d']:
        pos = np.array(json.load(open(md + f'{sid}_ses-01_hemi-{hemi}_desc-{roi}-wang_maskinfo.json'))['roiIndBold'])
        s = ok[pos]
        print(f"{hemi} {roi}: n={s.sum()}  mean x0={x[pos][s].mean():.2f}  mean y0={y[pos][s].mean():.2f}")

L V1v: n=925  mean x0=3.18  mean y0=-0.61
L V1d: n=774  mean x0=2.23  mean y0=2.73
R V1v: n=804  mean x0=-2.90  mean y0=-0.53
R V1d: n=998  mean x0=-1.87  mean y0=2.23


In [14]:
help(cortex.mask_flatmap)

Help on method mask_flatmap in module neuropythy.geometry.mesh:

mask_flatmap(mask, map_right='anterior', radius=Ellipsis, method=Ellipsis, pre_affine=Ellipsis, post_affine=Ellipsis, meta_data=Ellipsis, registration='native') method of neuropythy.mri.core.Cortex instance
    hemi.mask_flatmap(mask) yields a flatmap of the given hemisphere that is centered on the
      middle of the given mask.
    
    The following options may be given:
      * radius (default: Ellipsis) is passed along to the map_projection() function.
      * method (default: Ellipsis) is passed along to the map_projection() function.
      * pre_affine (default: Ellipsis) is passed along to the map_projection() function.
      * post_affine (default: Ellipsis) is passed along to the map_projection() function.
      * meta_data (default: Ellipsis) is passed along to the map_projection() function.
      * map_right (default: 'anterior') may be a direction ('anterior', 'posterior', 'inferior',
        'superior', 'lef

In [15]:
import glob, json, os, numpy as np, neuropythy as ny
sid, ses = 'sub-04', 'ses-01'
fs = f'/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/freesurfer-with_t2/{sid}'
base = f'/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/prfprepare/analysis-02_votcROI/{sid}/{ses}/func'
sub = ny.freesurfer_subject(fs)
for hc, h in [('L', 'lh'), ('R', 'rh')]:
    cx = sub.hemis[h]
    n = cx.vertex_count
    u, v = cx.tess.indexed_edges
    rois = {}
    for f in sorted(glob.glob(f'{base}/{sid}_{ses}_hemi-{hc}_desc-*-wang_maskinfo.json')):
        name = os.path.basename(f).split('desc-')[1].replace('-wang_maskinfo.json', '')
        m = np.zeros(n, bool)
        m[json.load(open(f))['roiIndFsnative']] = True
        rois[name] = m
    print(hc, 'ROIs found:', list(rois))
    for a in ['V3a', 'V3b']:
        touching = {b: int(np.sum((rois[a][u] & rois[b][v]) | (rois[b][u] & rois[a][v])))
                    for b in rois if b != a}
        print(hc, a, 'edges shared with:', {b: c for b, c in touching.items() if c > 0})

L ROIs found: ['IOG-words', 'IPS0', 'IPS1', 'IPS2', 'IPS3', 'IPS4', 'IPS5', 'LO1', 'LO2', 'PHC1', 'PHC2', 'PON-words', 'SPL1', 'TO1', 'TO2', 'V1d', 'V1v', 'V2d', 'V2v', 'V3a', 'V3b', 'V3d', 'V3v', 'VO1', 'VO2', 'hFEF', 'hV4', 'mFus-words', 'mOTS-words', 'pOTS-words']
L V3a edges shared with: {'V3b': 13}
L V3b edges shared with: {'LO1': 59, 'V3a': 13}
R ROIs found: ['IPS0', 'IPS1', 'IPS2', 'IPS3', 'IPS4', 'IPS5', 'LO1', 'LO2', 'PHC1', 'PHC2', 'SPL1', 'TO1', 'TO2', 'V1d', 'V1v', 'V2d', 'V2v', 'V3a', 'V3b', 'V3d', 'V3v', 'VO1', 'VO2', 'hFEF', 'hV4']
R V3a edges shared with: {'V3b': 55}
R V3b edges shared with: {'LO1': 75, 'V3a': 55}


In [16]:
import glob, json, os, numpy as np, neuropythy as ny
sid, ses = 'sub-04', 'ses-01'
fs = f'/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/freesurfer-with_t2/{sid}'
base = f'/bcbl/home/public/Gari/VOTCLOC/main_exp/BIDS/derivatives/prfprepare/analysis-02_votcROI/{sid}/{ses}/func'
wang = ['V1v','V1d','V2v','V2d','V3v','V3d','hV4','VO1','VO2','PHC1','PHC2','TO2','TO1',
        'LO2','LO1','V3b','V3a','IPS0','IPS1','IPS2','IPS3','IPS4','IPS5','SPL1','hFEF']
sub = ny.freesurfer_subject(fs)
for hc, h in [('L', 'lh'), ('R', 'rh')]:
    cx = sub.hemis[h]
    n = cx.vertex_count
    u, v = cx.tess.indexed_edges
    rois = {}
    for name in wang:
        f = f'{base}/{sid}_{ses}_hemi-{hc}_desc-{name}-wang_maskinfo.json'
        if os.path.isfile(f):
            m = np.zeros(n, bool)
            m[json.load(open(f))['roiIndFsnative']] = True
            rois[name] = m
    def touch(a):
        t = {b: int(np.sum((rois[a][u] & rois[b][v]) | (rois[b][u] & rois[a][v])))
             for b in rois if b != a}
        return {b: c for b, c in t.items() if c > 0}
    unlabeled = ~np.any(list(rois.values()), axis=0)
    def grow(seed, steps=15):
        cur = seed.copy()
        for _ in range(steps):
            nxt = cur.copy()
            nxt[v[cur[u] & unlabeled[v]]] = True
            nxt[u[cur[v] & unlabeled[u]]] = True
            cur = nxt
        return cur & unlabeled
    gap = grow(rois['V3d']) & grow(rois['IPS0'])
    print(hc, 'V3d touches:', touch('V3d'))
    print(hc, 'IPS0 touches:', touch('IPS0'))
    print(hc, 'unlabeled vertices reachable from both V3d and IPS0:', int(gap.sum()))

L V3d touches: {'V2d': 131, 'TO2': 93, 'TO1': 20, 'LO2': 64}
L IPS0 touches: {'TO2': 74, 'TO1': 37, 'IPS1': 137}
L unlabeled vertices reachable from both V3d and IPS0: 140
R V3d touches: {'V2d': 176, 'TO2': 97, 'TO1': 22, 'LO2': 67}
R IPS0 touches: {'TO2': 99, 'TO1': 49, 'IPS1': 154}
R unlabeled vertices reachable from both V3d and IPS0: 105
